In [ ]:
"""
FEATURE STABILITY ANALYSIS ACROSS REPEATED RESAMPLING
"Are these 46 genes stable?"
====================================================================

DESIGN PRINCIPLE:
  The 46-gene reference panel is NOT loaded from an old file -- it is
  RE-DERIVED here directly from your verified frozen_model.pkl (the same
  artifact already confirmed to match Supplementary Table S1 exactly and
  fit on all 130 GSE45827 samples). This guarantees the stability analysis
  tests the stability of the ACTUAL genes reported in the manuscript, with
  zero risk of drift from an out-of-sync saved list.
"""

# ============================================================
#  — Install dependencies
# ============================================================
!pip install GEOparse xgboost scikit-learn pandas numpy shap mygene -q

# ============================================================
#  — Imports
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import pickle

import GEOparse
import numpy as np
import pandas as pd
import shap
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

VALID_SUBTYPES = ["Basal", "Her2", "Luminal A", "Luminal B"]
SEED = 42
K_FEATURES = 300
N_SPLITS = 5
N_REPEATS = 4  # 5 x 4 = 20 total resampling runs, matching prior analysis scale

# Exact Supplementary Table S1 hyperparameters — used for EVERY model fit below
XGB_PARAMS = dict(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.3,
    subsample=1.0,
    colsample_bytree=1.0,
    eval_metric="mlogloss",
    objective="multi:softprob",
    random_state=SEED,
)

# ============================================================
#  — Load GSE45827 (discovery cohort, n=130)
# ============================================================


def load_gse45827():
    gse = GEOparse.get_GEO(geo="GSE45827", destdir="./geo_cache")
    rows, labels, sample_ids = [], [], []
    for gsm_name, gsm in gse.gsms.items():
        chars = gsm.metadata.get("characteristics_ch1", [])
        subtype = None
        for c in chars:
            if c.lower().startswith("tumor subtype"):
                subtype = c.split(":", 1)[1].strip()
        if subtype not in VALID_SUBTYPES:
            continue
        expr = gsm.table.set_index("ID_REF")["VALUE"]
        rows.append(expr)
        labels.append(subtype)
        sample_ids.append(gsm_name)
    X = pd.DataFrame(rows, index=sample_ids)
    y = pd.Series(labels, index=sample_ids, name="subtype")
    return X, y


print("Loading GSE45827...")
X, y = load_gse45827()
print(f"Samples: {X.shape[0]}, probes: {X.shape[1]}")

# ============================================================
#  — Load verified frozen artifacts
# ============================================================

import pickle

with open("model/frozen_selector.pkl", "rb") as f:
    frozen_selector = pickle.load(f)
with open("model/frozen_scaler.pkl", "rb") as f:
    frozen_scaler = pickle.load(f)
with open("model/frozen_model.pkl", "rb") as f:
    frozen_model = pickle.load(f)
with open("model/frozen_label_encoder.pkl", "rb") as f:
    frozen_le = pickle.load(f)
with open("model/selected_probe_ids.pkl", "rb") as f:
    frozen_probe_ids = pickle.load(f)

print(f"Loaded frozen artifacts. Frozen model uses {len(frozen_probe_ids)} probes.")
print("Frozen model params (spot check):", frozen_model.get_params()["n_estimators"],
      frozen_model.get_params()["max_depth"], frozen_model.get_params()["learning_rate"])

y_encoded_full = frozen_le.transform(y)

# ============================================================
#  — Re-derive the canonical 46-gene reference panel from the
# VERIFIED frozen model (matches manuscript Methods 2.3 exactly)
# ============================================================

X_frozen_selected = X[frozen_probe_ids].values
X_frozen_scaled = frozen_scaler.transform(X_frozen_selected)

explainer_ref = shap.TreeExplainer(frozen_model)
shap_values_ref = explainer_ref.shap_values(X_frozen_scaled)


def mean_abs_shap(shap_values):
    """Normalize SHAP output across shap-library versions into a
    (n_features,) importance vector, averaged across classes and samples."""
    arr = np.array(shap_values)
    if arr.ndim == 3:
        # Could be (n_classes, n_samples, n_features) or (n_samples, n_features, n_classes)
        if arr.shape[0] < arr.shape[1] and arr.shape[0] < arr.shape[2]:
            return np.abs(arr).mean(axis=(0, 1))  # (n_classes, n_samples, n_features)
        else:
            return np.abs(arr).mean(axis=(0, 2))  # (n_samples, n_features, n_classes)
    return np.abs(arr).mean(axis=0)  # already (n_samples, n_features)


ref_importance = mean_abs_shap(shap_values_ref)
ref_ranking = pd.Series(ref_importance, index=frozen_probe_ids).sort_values(ascending=False)
top50_probes = ref_ranking.head(50).index.tolist()

print(f"Top 50 probes by mean |SHAP| identified from frozen model.")

# Map probes -> HGNC gene symbols (matches manuscript Methods 2.1/2.3)
try:
    import mygene
    mg = mygene.MyGeneInfo()
    query_res = mg.querymany(top50_probes, scopes="reporter", fields="symbol", species="human")
    probe_to_symbol = {}
    for r in query_res:
        if "symbol" in r:
            probe_to_symbol[r["query"]] = r["symbol"]
except Exception as e:
    print(f"mygene mapping failed ({e}) -- proceeding with probe IDs only.")
    probe_to_symbol = {}

reference_panel_df = pd.DataFrame({
    "probe_id": top50_probes,
    "gene_symbol": [probe_to_symbol.get(p, np.nan) for p in top50_probes],
    "shap_importance": ref_ranking.head(50).values,
})
reference_panel_df["rank"] = range(1, len(reference_panel_df) + 1)

n_unique_genes = reference_panel_df["gene_symbol"].nunique()
print(f"Resolved to {n_unique_genes} unique gene symbols "
      f"(manuscript reports 46 after resolving 4 unmapped probes).")

reference_panel_df.to_csv("reference_46gene_panel.csv", index=False)
print("Saved: reference_46gene_panel.csv  <-- this IS 'the 46 genes' for this analysis")

# ============================================================
#  — Repeated resampling: refit selector/scaler/model PER FOLD
# (no leakage), track selection + SHAP importance of every probe each run
# ============================================================

cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)

run_records = []       # per-probe, per-run: selected / importance / rank
perf_records = []      # per-run: accuracy, macro-F1

for run_idx, (train_idx, test_idx) in enumerate(cv.split(X.values, y_encoded_full)):
    X_tr, X_te = X.values[train_idx], X.values[test_idx]
    y_tr, y_te = y_encoded_full[train_idx], y_encoded_full[test_idx]

    selector = SelectKBest(score_func=mutual_info_classif, k=K_FEATURES)
    scaler = StandardScaler()
    model = XGBClassifier(**XGB_PARAMS)

    X_tr_sel = selector.fit_transform(X_tr, y_tr)
    X_tr_scaled = scaler.fit_transform(X_tr_sel)
    model.fit(X_tr_scaled, y_tr)

    X_te_sel = selector.transform(X_te)
    X_te_scaled = scaler.transform(X_te_sel)
    y_pred = model.predict(X_te_scaled)

    perf_records.append({
        "run": run_idx,
        "test_accuracy": accuracy_score(y_te, y_pred),
        "test_macro_f1": f1_score(y_te, y_pred, average="macro"),
    })

    run_probe_ids = X.columns[selector.get_support()].tolist()

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_tr_scaled)
    importance = mean_abs_shap(shap_values)
    ranking = pd.Series(importance, index=run_probe_ids).sort_values(ascending=False)
    rank_lookup = {probe: rank for rank, probe in enumerate(ranking.index, start=1)}

    for probe in run_probe_ids:
        run_records.append({
            "run": run_idx,
            "probe_id": probe,
            "importance": ranking[probe],
            "rank_in_run": rank_lookup[probe],
        })

    print(f"Run {run_idx + 1}/{N_SPLITS * N_REPEATS}: "
          f"test_acc={perf_records[-1]['test_accuracy']:.3f}, "
          f"test_macro_f1={perf_records[-1]['test_macro_f1']:.3f}")

runs_df = pd.DataFrame(run_records)
perf_df = pd.DataFrame(perf_records)

# ============================================================
#  — Aggregate stability stats for EVERY probe ever selected
# ============================================================

n_total_runs = N_SPLITS * N_REPEATS

agg = runs_df.groupby("probe_id").agg(
    times_selected=("run", "count"),
    mean_importance=("importance", "mean"),
    median_importance=("importance", "median"),
    mean_rank=("rank_in_run", "mean"),
    best_rank=("rank_in_run", "min"),
).reset_index()
agg["selection_frequency_pct"] = 100 * agg["times_selected"] / n_total_runs

# Map every probe here to gene symbol too (reuse Cell 5 mapping + extend if needed)
all_probes_seen = agg["probe_id"].tolist()
missing_mapping = [p for p in all_probes_seen if p not in probe_to_symbol]
if missing_mapping:
    try:
        query_res2 = mg.querymany(missing_mapping, scopes="reporter", fields="symbol", species="human")
        for r in query_res2:
            if "symbol" in r:
                probe_to_symbol[r["query"]] = r["symbol"]
    except Exception as e:
        print(f"Extended mygene mapping failed ({e}).")

agg["gene_symbol"] = agg["probe_id"].map(probe_to_symbol)
agg = agg.sort_values("selection_frequency_pct", ascending=False)
agg.to_csv("all_probes_stability_full.csv", index=False)
print(f"\n{len(agg)} distinct probes were selected at least once across {n_total_runs} runs.")
print("Saved: all_probes_stability_full.csv (complete, unfiltered)")

# ============================================================
#  — Stability of the 46 REFERENCE panel genes,
# specifically, by name — 
# ============================================================

reference_probes = reference_panel_df["probe_id"].tolist()

panel_stability = reference_panel_df[["probe_id", "gene_symbol", "rank"]].merge(
    agg[["probe_id", "times_selected", "selection_frequency_pct",
         "mean_importance", "mean_rank", "best_rank"]],
    on="probe_id", how="left"
)
# Genes never re-selected in any of the 20 runs won't appear in `agg` — fill explicitly
panel_stability["times_selected"] = panel_stability["times_selected"].fillna(0).astype(int)
panel_stability["selection_frequency_pct"] = panel_stability["selection_frequency_pct"].fillna(0.0)

panel_stability = panel_stability.rename(columns={"rank": "original_shap_rank"})
panel_stability = panel_stability.sort_values("selection_frequency_pct", ascending=False)
panel_stability.to_csv("46gene_panel_stability.csv", index=False)

print("\n=== Stability of the 46-gene SHAP panel (PRIMARY RESULT) ===")
print(panel_stability.to_string(index=False))
print("\nSaved: 46gene_panel_stability.csv")

n_50pct_plus = (panel_stability["selection_frequency_pct"] >= 50).sum()
median_freq = panel_stability["selection_frequency_pct"].median()
print(f"\n{n_50pct_plus} of {len(panel_stability)} panel genes selected in >=50% of runs.")
print(f"Median selection frequency across the full 46-gene panel: {median_freq:.1f}%")

# Explicitly surface CDCA5 and CMC2 regardless of rank
for gene_name in ["CDCA5", "CMC2"]:
    row = panel_stability[panel_stability["gene_symbol"] == gene_name]
    if len(row):
        r = row.iloc[0]
        print(f"\n{gene_name}: selected in {int(r['times_selected'])}/{n_total_runs} runs "
              f"({r['selection_frequency_pct']:.1f}%), "
              f"mean rank when selected = {r['mean_rank']:.1f}, best rank = {int(r['best_rank']) if pd.notna(r['best_rank']) else 'N/A'}")
    else:
        print(f"\n{gene_name}: NOT FOUND in reference panel mapping -- check gene_symbol mapping manually.")

# ============================================================
#  — SECONDARY view: top 20 most stable genes overall (unconstrained)
# Complementary context only -
# ============================================================

top20_overall = agg.head(20)[["probe_id", "gene_symbol", "times_selected",
                                "selection_frequency_pct", "mean_importance", "mean_rank"]]
top20_overall.to_csv("top20_stable_genes_overall.csv", index=False)
print("\n=== SECONDARY: Top 20 most stable genes overall (all probes, unconstrained) ===")
print(top20_overall.to_string(index=False))
print("\n(Note: this may include genes outside the original 46-gene panel, and is provided")
print("as context only -- the primary answer to 'are these 46 genes stable' is Cell 8's table.)")

# ============================================================
#  — Model performance robustness across the 20 resampling runs
# ============================================================

perf_summary = pd.DataFrame({
    "metric": ["n_runs", "mean_test_accuracy", "sd_test_accuracy",
               "mean_test_macro_f1", "sd_test_macro_f1",
               "min_test_accuracy", "max_test_accuracy"],
    "value": [
        n_total_runs,
        perf_df["test_accuracy"].mean(), perf_df["test_accuracy"].std(),
        perf_df["test_macro_f1"].mean(), perf_df["test_macro_f1"].std(),
        perf_df["test_accuracy"].min(), perf_df["test_accuracy"].max(),
    ],
})
perf_summary.to_csv("stability_performance_summary.csv", index=False)
print("\n=== Model performance across repeated resampling ===")
print(perf_summary.to_string(index=False))
print("\nSaved: stability_performance_summary.csv")

print("\n" + "=" * 70)
print("SUMMARY FOR MANUSCRIPT / RESPONSE TO MAM:")
print(f"  {n_50pct_plus}/46 panel genes selected in >=50% of {n_total_runs} resampling runs.")
print(f"  Median panel selection frequency: {median_freq:.1f}%.")
print("  CDCA5 and CMC2 stability reported explicitly above, by name.")
print("=" * 70)

09-Jul-2026 16:53:29 DEBUG utils - Directory ./geo_cache already exists. Skipping.
09-Jul-2026 16:53:29 INFO GEOparse - File already exist: using local version.
09-Jul-2026 16:53:29 INFO GEOparse - Parsing ./geo_cache/GSE45827_family.soft.gz: 
09-Jul-2026 16:53:29 DEBUG GEOparse - DATABASE: GeoMiame
09-Jul-2026 16:53:29 DEBUG GEOparse - SERIES: GSE45827
09-Jul-2026 16:53:29 DEBUG GEOparse - PLATFORM: GPL570


Loading GSE45827...


09-Jul-2026 16:53:31 DEBUG GEOparse - SAMPLE: GSM1116084
09-Jul-2026 16:53:31 DEBUG GEOparse - SAMPLE: GSM1116085
09-Jul-2026 16:53:31 DEBUG GEOparse - SAMPLE: GSM1116086
09-Jul-2026 16:53:31 DEBUG GEOparse - SAMPLE: GSM1116087
09-Jul-2026 16:53:31 DEBUG GEOparse - SAMPLE: GSM1116088
09-Jul-2026 16:53:31 DEBUG GEOparse - SAMPLE: GSM1116089
09-Jul-2026 16:53:31 DEBUG GEOparse - SAMPLE: GSM1116090
09-Jul-2026 16:53:31 DEBUG GEOparse - SAMPLE: GSM1116091
09-Jul-2026 16:53:31 DEBUG GEOparse - SAMPLE: GSM1116092
09-Jul-2026 16:53:31 DEBUG GEOparse - SAMPLE: GSM1116093
09-Jul-2026 16:53:31 DEBUG GEOparse - SAMPLE: GSM1116094
09-Jul-2026 16:53:31 DEBUG GEOparse - SAMPLE: GSM1116095
09-Jul-2026 16:53:31 DEBUG GEOparse - SAMPLE: GSM1116096
09-Jul-2026 16:53:31 DEBUG GEOparse - SAMPLE: GSM1116097
09-Jul-2026 16:53:31 DEBUG GEOparse - SAMPLE: GSM1116098
09-Jul-2026 16:53:31 DEBUG GEOparse - SAMPLE: GSM1116099
09-Jul-2026 16:53:31 DEBUG GEOparse - SAMPLE: GSM1116100
09-Jul-2026 16:53:31 DEBUG GEOp

Samples: 130, probes: 29873
Loaded frozen artifacts. Frozen model uses 300 probes.
Frozen model params (spot check): 100 6 0.3
Top 50 probes by mean |SHAP| identified from frozen model.


4 input query terms found no hit:	['229150_at', '215593_at', '233445_at', '242580_at']


Resolved to 43 unique gene symbols (manuscript reports 46 after resolving 4 unmapped probes).
Saved: reference_46gene_panel.csv  <-- this IS 'the 46 genes' for this analysis
Run 1/20: test_acc=0.846, test_macro_f1=0.825
Run 2/20: test_acc=0.923, test_macro_f1=0.917
Run 3/20: test_acc=0.962, test_macro_f1=0.963
Run 4/20: test_acc=0.923, test_macro_f1=0.914
Run 5/20: test_acc=1.000, test_macro_f1=1.000
Run 6/20: test_acc=0.885, test_macro_f1=0.873
Run 7/20: test_acc=0.923, test_macro_f1=0.921
Run 8/20: test_acc=0.923, test_macro_f1=0.916
Run 9/20: test_acc=0.923, test_macro_f1=0.922
Run 10/20: test_acc=0.885, test_macro_f1=0.877
Run 11/20: test_acc=0.962, test_macro_f1=0.955
Run 12/20: test_acc=0.962, test_macro_f1=0.958
Run 13/20: test_acc=0.769, test_macro_f1=0.703
Run 14/20: test_acc=1.000, test_macro_f1=1.000
Run 15/20: test_acc=0.923, test_macro_f1=0.919
Run 16/20: test_acc=1.000, test_macro_f1=1.000
Run 17/20: test_acc=0.962, test_macro_f1=0.958
Run 18/20: test_acc=0.962, test_macr

23 input query terms found dup hits:	[('1563591_at', 3), ('202233_s_at', 2), ('204060_s_at', 2), ('204650_s_at', 2), ('204962_s_at', 2), 
66 input query terms found no hit:	['1556345_s_at', '1556923_at', '1557452_at', '1558447_at', '1558448_a_at', '1559078_at', '1560869_a_



769 distinct probes were selected at least once across 20 runs.
Saved: all_probes_stability_full.csv (complete, unfiltered)

=== Stability of the 46-gene SHAP panel (PRIMARY RESULT) ===
    probe_id gene_symbol  original_shap_rank  times_selected  selection_frequency_pct  mean_importance  mean_rank  best_rank
 218211_s_at        MLPH                   1              20                    100.0         0.272021  14.950000          1
 210930_s_at       ERBB2                   3              20                    100.0         0.276958   3.000000          1
 210761_s_at        GRB7                  17              20                    100.0         0.054946  36.000000          1
   205225_at        ESR1                   4              20                    100.0         0.150127  16.050000          1
 234354_x_at       ERBB2                   5              20                    100.0         0.166333  25.400000          2
 232322_x_at     STARD10                   6              20   

In [2]:
TOP_K_STRICT = 50  # matches how the original 46-gene panel was derived

runs_df["in_top50_this_run"] = runs_df["rank_in_run"] <= TOP_K_STRICT

strict_agg = runs_df[runs_df["in_top50_this_run"]].groupby("probe_id").agg(
    times_in_top50=("run", "count"),
).reset_index()
strict_agg["top50_frequency_pct"] = 100 * strict_agg["times_in_top50"] / n_total_runs

panel_strict = reference_panel_df[["probe_id", "gene_symbol", "rank"]].merge(
    strict_agg, on="probe_id", how="left"
)
panel_strict["times_in_top50"] = panel_strict["times_in_top50"].fillna(0).astype(int)
panel_strict["top50_frequency_pct"] = panel_strict["top50_frequency_pct"].fillna(0.0)
panel_strict = panel_strict.rename(columns={"rank": "original_shap_rank"}).sort_values(
    "top50_frequency_pct", ascending=False
)
panel_strict.to_csv("46gene_panel_stability_STRICT_top50.csv", index=False)
print(panel_strict.to_string(index=False))

n50plus_strict = (panel_strict["top50_frequency_pct"] >= 50).sum()
print(f"\n{n50plus_strict} of {len(panel_strict)} genes landed in the run's own top-{TOP_K_STRICT} SHAP ranking in >=50% of runs.")

for gene_name in ["CDCA5", "CMC2"]:
    row = panel_strict[panel_strict["gene_symbol"] == gene_name]
    if len(row):
        r = row.iloc[0]
        print(f"{gene_name}: in top-{TOP_K_STRICT} in {int(r['times_in_top50'])}/{n_total_runs} runs ({r['top50_frequency_pct']:.1f}%)")

    probe_id gene_symbol  original_shap_rank  times_in_top50  top50_frequency_pct
 232322_x_at     STARD10                   6              20                100.0
 210930_s_at       ERBB2                   3              20                100.0
   205225_at        ESR1                   4              19                 95.0
   220054_at       IL23A                  15              19                 95.0
   213611_at        AQP5                  13              18                 90.0
 218211_s_at        MLPH                   1              18                 90.0
   200934_at         DEK                   8              18                 90.0
 210761_s_at        GRB7                  17              18                 90.0
   221811_at       PGAP3                  11              18                 90.0
 234354_x_at       ERBB2                   5              18                 90.0
 204092_s_at       AURKA                   2              17                 85.0
    55616_at    

In [4]:
panel_strict.to_csv('/kaggle/working/strict_top50_stability_gse45827.csv', index=False)
print("Saved: /kaggle/working/strict_top50_stability_gse45827.csv")

Saved: /kaggle/working/strict_top50_stability_gse45827.csv
